## Setup

In [ ]:
!pip install uv
!git clone -b scratch https://github.com/estiftole/c-swm.git

# Move into the folder
import os
import sys

os.chdir('c-swm')
sys.path.append('.')

# Fix python ver
!sed -i 's/>=3.13/>=3.12/g' pyproject.toml
# Install deps
!uv pip install --system -r pyproject.toml

In [ ]:
num_train_episodes = 1000
num_eval_episodes = 100
num_epochs = 200
batch_size = 64
seeds = [42, 43, 44, 45, 46]
print(f"> {num_train_episodes} episodes for training data\n> {num_eval_episodes} episodes for evaluation data\nTraining for {num_epochs} epochs")

## Data Generation

In [ ]:
# Generate pong training and eval data
!uv run gen_data.py --env_id ALE/Pong-v5 --fname data/pong_train.h5 --num_episodes {num_train_episodes} --atari --seed 1
!uv run gen_data.py --env_id ALE/Pong-v5 --fname data/pong_eval.h5 --num_episodes {num_eval_episodes} --atari --seed 2

In [ ]:
# Generate breakout training and eval data
!uv run gen_data.py --env_id ALE/Breakout-v5 --fname data/breakout_train.h5 --num_episodes {num_train_episodes} --atari --seed 1
!uv run gen_data.py --env_id ALE/Breakout-v5 --fname data/breakout_eval.h5 --num_episodes {num_eval_episodes} --atari --seed 2

In [ ]:
# Generate centipede training and eval data
!uv run gen_data.py --env_id ALE/Centipede-v5 --fname data/centipede_train.h5 --num_episodes {num_train_episodes} --atari --seed 1
!uv run gen_data.py --env_id ALE/Centipede-v5 --fname data/centipede_eval.h5 --num_episodes {num_eval_episodes} --atari --seed 2

## Train and Eval C-SWM Model

In [ ]:
# Train and evaluate Pong
for seed in seeds:
    !uv run train.py --dataset data/pong_train.h5 --embedding-dim 4 --action-dim 6 --num-slots 3 --batch-size {batch_size} --global-action --epochs {num_epochs} --name pong_cswm --seed {seed}
    for n_steps in [1,5,10]:
        !uv run eval.py --save-folder checkpoints/pong_cswm --dataset data/pong_eval.h5 --save-folder checkpoints/pong_cswm --num-steps {n_steps}

In [ ]:
# Train and evaluate Breakout
for seed in seeds:
    !uv run train.py --dataset data/breakout_train.h5 --embedding-dim 4 --action-dim 4 --num-slots 5 --batch-size {batch_size} --global-action --epochs {num_epochs} --name breakout_cswm --seed {seed}
    for n_steps in [1,5,10]:
        !uv run eval.py --save-folder checkpoints/breakout_cswm --dataset data/breakout_eval.h5 --save-folder checkpoints/breakout_cswm --num-steps {n_steps}

In [ ]:
# Train and evaluate Centipede
for seed in seeds:
    !uv run train.py --dataset data/centipede_train.h5 --embedding-dim 8 --action-dim 18 --num-slots 8 --batch-size {batch_size} --global-action --epochs {num_epochs} --name centipede_cswm --seed {seed}
    for n_steps in [1,5,10]:
        !uv run eval.py --save-folder checkpoints/centipede_cswm --dataset data/centipede_eval.h5 --num-steps {n_steps}

## Train and Eval Reconstruction-based Model

In [ ]:
# Train and evaluate Pong
for seed in seeds:
    !uv run train.py --dataset data/pong_train.h5 --embedding-dim 4 --action-dim 6 --num-slots 3 --batch-size {batch_size} --global-action --epochs {num_epochs} --seed {seed} --decoder --name pong_recon
    for n_steps in [1,5,10]:
        !uv run eval.py --dataset data/pong_eval.h5 --save-folder checkpoints/pong_recon --num-steps {n_steps}

In [ ]:
# Train and evaluate Breakout
for seed in seeds:
    !uv run train.py --dataset data/breakout_train.h5 --embedding-dim 4 --action-dim 4 --num-slots 5 --batch-size {batch_size} --global-action --epochs {num_epochs} --name breakout_recon --decoder --seed {seed}
    for n_steps in [1,5,10]:
        !uv run eval.py --dataset data/breakout_eval.h5 --save-folder checkpoints/breakout_recon --num-steps {n_steps}

In [ ]:
# Train and evaluate Centipede
for seed in seeds:
    !uv run train.py --dataset data/centipede_train.h5 --embedding-dim 8 --action-dim 18 --num-slots 8 --batch-size {batch_size} --global-action --epochs {num_epochs} --name centipede_recon --decoder --seed {seed}
    for n_steps in [1,5,10]:
        !uv run eval.py --dataset data/centipede_eval.h5 --save-folder checkpoints/centipede_recon --num-steps {n_steps}

## Compare Performance

In [ ]:
import json
import pandas as pd

records = []
with open("data/cswm-results.jsonl", "r") as f:
    for line in f:
        data = json.loads(line)
        records.append({
            "model_name": data["model_name"],
            "decoder": data["decoder"],
            "seed": data["seed"],
            "num_steps": data["num_steps"],
            "hits_at_1": data["hits_at_1"],
            "mrr": data["mrr"],
        })

df = pd.DataFrame(records)

stats = df.groupby(["model_name", "decoder", "num_steps"]).agg(
    hits_mean=("hits_at_1", "mean"),
    hits_std=("hits_at_1", "std"),
    mrr_mean=("mrr", "mean"),
    mrr_std=("mrr", "std"),
).reset_index()

stats["Hits@1"] = stats.apply(lambda r: f"{r['hits_mean']:.3f} ± {r['hits_std']:.3f}", axis=1)
stats["MRR"] = stats.apply(lambda r: f"{r['mrr_mean']:.3f} ± {r['mrr_std']:.3f}", axis=1)

summary_table = stats[["model_name", "decoder", "num_steps", "Hits@1", "MRR"]]
print(summary_table.to_string(index=False))
